# GPS-Denied Vehicle Dead Reckoning: Reproducible Google Colab GPU Benchmark

**Project:** IO-VNBD Smartphone Inertial Navigation Benchmark  
**Target:** Continuous Forward Speed Estimation with Gyroscope Azimuth Integration  
**Architectures:** CNN-GRU Baseline, ModernTCN, Mamba-2 (vectorized State Space Duality)  
**Execution:** GPU-accelerated (CUDA) with automated CPU fallback and Google Drive persistence  

---

### Experimental Protocol
1. **Input:** 6 phone IMU channels (`accel_x`, `accel_y`, `accel_z`, `gyro_yaw`, `gyro_pitch`, `gyro_roll`) over 10-second sliding windows (100 samples at 10 Hz).
2. **Target:** Instantaneous forward speed $s(t) = v_{\text{fwd}}(t)$ in m/s from high-rate vehicle CAN/OBD telemetry.
3. **Grouped Splits (Zero Leakage):**
   - **Train:** Sept 8, 2019 (Drives S1 + S2, 14,525 windows)
   - **Val:** Sept 4, 2019 (Drives S3a + S3b + S3c, 6,812 windows)
   - **Test:** Sept 6, 2019 (Drive S4, 9,117 windows holdout)
4. **Outage Simulation:** Causal trajectory dead reckoning over 10s, 30s, and 60s simulated GNSS outages.

## Cell 1: Hardware Diagnostics & GPU Detection

In [ ]:
!nvidia-smi

import torch
print("=" * 60)
print("PYTORCH & ACCELERATOR DIAGNOSTICS")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("Running on CPU (CUDA not available)")
print("=" * 60)

## Cell 2: Mount Google Drive & Set Persistent Paths

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Set your project path on Google Drive
    DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/GPS_Denied_Navigation_Research')
    DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Mounted Google Drive. Persistent outputs will save to: {DRIVE_PROJECT_ROOT}")
except ImportError:
    DRIVE_PROJECT_ROOT = Path.cwd()
    print(f"Running outside Google Colab. Using working directory: {DRIVE_PROJECT_ROOT}")

## Cell 3: Setup Local Fast NVMe Cache Storage
To avoid slow Google Drive FUSE latency during data loading, copy processed dataset tensors to `/content/data/processed` on the local VM NVMe SSD.

In [ ]:
from scripts.colab_runner import setup_local_storage

# Source processed datasets (Drive or repo root)
source_dir = Path('data/processed')
if not source_dir.exists() and (DRIVE_PROJECT_ROOT / 'data/processed').exists():
    source_dir = DRIVE_PROJECT_ROOT / 'data/processed'

local_cache_dir = Path('/content/data/processed') if os.path.exists('/content') else Path('data/processed')
cached_dir = setup_local_storage(source_dir, local_cache_dir)

## Cell 4: Install Dependencies

In [ ]:
!pip install -q pyyaml pandas numpy matplotlib scipy pytest

## Cell 5: Run Unit Test Suite
Verify geodesy, coordinate transformations, synchronization, normalizer train-only isolation, and data causality.

In [ ]:
!pytest -v tests/

## Cell 6: Numerical Consistency & Checkpoint-Resume Verification
Verifies determinism across devices and proves exact state restoration from `last_checkpoint.pt`.

In [ ]:
!python scripts/verify_numerical_consistency.py

## Cell 7: Fast GPU Smoke Test
Runs a fast 1-epoch sanity check across CNN-GRU, ModernTCN, and Mamba-2 on GPU before beginning the full multi-seed benchmark.

In [ ]:
!python scripts/colab_runner.py --mode smoke_test --device auto --local_cache_dir /content/data/processed

## Cell 8: Full 9-Model GPU Benchmark Execution
Trains CNN-GRU, ModernTCN, and Mamba-2 across seeds 42, 123, 456 on GPU with per-epoch checkpoints saved to Google Drive.

In [ ]:
!python scripts/run_phase8_training.py \
    --device auto \
    --data_dir /content/data/processed \
    --output_dir /content/drive/MyDrive/GPS_Denied_Navigation_Research/checkpoints \
    --seeds 42 123 456

## Cell 9: Outage Dead-Reckoning Trajectory Benchmark & Profiling
Evaluates speed estimation accuracy on holdout Drive S4 and executes step-by-step causal dead reckoning across 10s, 30s, and 60s GNSS outages.

In [ ]:
!python scripts/evaluate_speed_models.py \
    --device auto \
    --data_dir /content/data/processed \
    --ckpt_dir /content/drive/MyDrive/GPS_Denied_Navigation_Research/checkpoints \
    --results_dir /content/drive/MyDrive/GPS_Denied_Navigation_Research/results

## Cell 10: Generate Figures & Display Results

In [ ]:
!python scripts/plot_loss_curves.py

import pandas as pd
from IPython.display import display, Image

speed_csv = Path('/content/drive/MyDrive/GPS_Denied_Navigation_Research/results/model_speed_metrics.csv')
if not speed_csv.exists():
    speed_csv = Path('experiments/phase8_corrected_speed_models/results/model_speed_metrics.csv')

outage_csv = Path('/content/drive/MyDrive/GPS_Denied_Navigation_Research/results/model_outage_metrics.csv')
if not outage_csv.exists():
    outage_csv = Path('experiments/phase8_corrected_speed_models/results/model_outage_metrics.csv')

print("=== SPEED ESTIMATION ACCURACY (HOLD-OUT DRIVE S4) ===")
df_spd = pd.read_csv(speed_csv)
display(df_spd[['model_run', 'model_type', 'param_count', 'val_mae', 'test_mae', 'test_rmse', 'latency_b1_ms']])

print("\n=== TRAJECTORY DEAD RECKONING ACCURACY (10s, 30s, 60s OUTAGES) ===")
df_out = pd.read_csv(outage_csv)
df_out['arch'] = df_out['method'].apply(lambda m: 'CNN-GRU' if 'cnngru' in m else ('ModernTCN' if 'moderntcn' in m else ('Mamba-2 SSD' if 'mamba2' in m else m)))
summary = df_out.groupby(['duration_s', 'arch'])[['ate_rmse', 'final_drift_m', 'drift_pct']].mean().round(2)
display(summary)

if os.path.exists('figures/phase8_training_loss_curves.png'):
    display(Image('figures/phase8_training_loss_curves.png'))

if os.path.exists('figures/phase8_outage_trajectories_60s.png'):
    display(Image('figures/phase8_outage_trajectories_60s.png'))